<a href="https://colab.research.google.com/github/KanitAon/ebay-soccer-card-market-analysis/blob/main/src/data_analysis/What_Affects_Soccer_Card_Selling_Prices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# All-in-One Football Card Price Analysis

This Google Colab notebook combines the five football-card price analyses into one workflow.

**Run order**

1. Upload the eBay market dataset **once** (`.csv`, `.xlsx`, or `.xls`).
2. Run all analysis sections.
3. Each section creates its graph and Excel output.
4. The final section packages all generated files into one ZIP and downloads it.

## Included analyses

1. Autograph / Patch price analysis
2. Panini vs Topps by card type
3. Graded vs Ungraded — matched by the same player and box set
4. Rookie vs Non-Rookie — matched by the same player and box set
5. Panini vs Topps asking price by product tier

The graphs do **not** display p-values. Statistical test results are shown in tables and exported to Excel.


## 0. Import Libraries and Create Output Folder

In [ ]:

import os
import shutil
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import ttest_ind, ttest_rel, mannwhitneyu, wilcoxon
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multitest import multipletests
from google.colab import files
from IPython.display import display

warnings.filterwarnings("ignore")

OUTPUT_DIR = "football_card_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Outputs will be saved in: {OUTPUT_DIR}/")


## 1. Upload the Dataset Once

In [ ]:

uploaded = files.upload()

data_files = [
    name for name in uploaded.keys()
    if name.lower().endswith((".csv", ".xlsx", ".xls"))
]

if not data_files:
    raise FileNotFoundError("No CSV or Excel file was uploaded.")

FILE_PATH = data_files[0]

if FILE_PATH.lower().endswith(".csv"):
    df_raw = pd.read_csv(FILE_PATH, low_memory=False)
else:
    df_raw = pd.read_excel(FILE_PATH)

print(f"Using file: {FILE_PATH}")
print(f"Rows: {len(df_raw):,}")
print(f"Columns: {len(df_raw.columns):,}")
display(df_raw.head())



## Helper Functions

These helpers keep the five analyses independent from each other, so cleaning in one section does not overwrite the original dataset.


In [ ]:

def require_columns(dataframe, required, analysis_name):
    missing = [col for col in required if col not in dataframe.columns]
    if missing:
        raise ValueError(
            f"{analysis_name} cannot run. Missing required columns: {missing}"
        )

def to_bool(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

def format_pvalue(value):
    if pd.isna(value):
        return "N/A"
    return f"{value:.2e}" if value < 0.001 else f"{value:.4f}"

generated_files = []
analysis_status = []



# Analysis 1 — Autograph & Patch Price Analysis

Compares four card-feature groups:

- No Autograph / Patch
- Autograph Only
- Patch Only
- Autograph + Patch

Outputs include summary statistics, a 500-DPI graph, Welch's ANOVA, Kruskal-Wallis, and pairwise Mann-Whitney U tests with Holm correction.


In [ ]:

def run_analysis_01(df_raw):
    analysis_name = "Analysis 1 - Autograph & Patch"
    df = df_raw.copy()

    require_columns(
        df,
        ["asking_price_usd", "autograph", "patch"],
        analysis_name
    )

    df["asking_price_usd"] = pd.to_numeric(
        df["asking_price_usd"], errors="coerce"
    )
    df = df[
        df["asking_price_usd"].notna() &
        (df["asking_price_usd"] > 0)
    ].copy()

    df["autograph"] = to_bool(df["autograph"])
    df["patch"] = to_bool(df["patch"])

    conditions = [
        (df["autograph"] == True) & (df["patch"] == True),
        (df["autograph"] == True) & (df["patch"] == False),
        (df["autograph"] == False) & (df["patch"] == True)
    ]

    choices = [
        "Autograph + Patch",
        "Autograph Only",
        "Patch Only"
    ]

    group_order = [
        "No Autograph / Patch",
        "Autograph Only",
        "Patch Only",
        "Autograph + Patch"
    ]

    df["card_type"] = np.select(
        conditions,
        choices,
        default="No Autograph / Patch"
    )

    df["card_type"] = pd.Categorical(
        df["card_type"],
        categories=group_order,
        ordered=True
    )

    summary = (
        df.groupby("card_type", observed=True)["asking_price_usd"]
        .agg(
            Listings="count",
            Average_Price="mean",
            Median_Price="median",
            Std_Dev="std",
            Min_Price="min",
            Max_Price="max"
        )
        .reindex(group_order)
        .reset_index()
    )

    summary[
        ["Average_Price", "Median_Price", "Std_Dev", "Min_Price", "Max_Price"]
    ] = summary[
        ["Average_Price", "Median_Price", "Std_Dev", "Min_Price", "Max_Price"]
    ].round(2)

    print("SUMMARY STATISTICS")
    display(summary)

    plot_data = (
        df.groupby("card_type", observed=True)["asking_price_usd"]
        .agg(Average_Price="mean", Listings="count")
        .reindex(group_order)
        .reset_index()
        .sort_values("Average_Price")
    )

    fig, ax = plt.subplots(figsize=(10, 5.8))
    bars = ax.barh(
        plot_data["card_type"],
        plot_data["Average_Price"]
    )

    for bar, price, n in zip(
        bars,
        plot_data["Average_Price"],
        plot_data["Listings"]
    ):
        ax.text(
            bar.get_width() + max(plot_data["Average_Price"].max() * 0.015, 1),
            bar.get_y() + bar.get_height() / 2,
            f"${price:,.0f}   |   {n:,} listings",
            va="center",
            fontsize=11,
            fontweight="bold"
        )

    ax.set_title(
        "Average Asking Price by Card Type",
        fontsize=17,
        fontweight="bold",
        pad=15
    )
    ax.set_xlabel("Average Asking Price (USD)")
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.2)

    if plot_data["Average_Price"].notna().any():
        ax.set_xlim(0, plot_data["Average_Price"].max() * 1.42)

    plt.tight_layout()

    graph_file = os.path.join(
        OUTPUT_DIR, "01_average_price_by_card_type.png"
    )
    plt.savefig(graph_file, dpi=500, bbox_inches="tight")
    plt.show()

    groups = {
        group: df.loc[
            df["card_type"] == group,
            "asking_price_usd"
        ].dropna().values
        for group in group_order
    }

    print("\nGROUP SIZES")
    for group, values in groups.items():
        mean_value = np.mean(values) if len(values) else np.nan
        median_value = np.median(values) if len(values) else np.nan
        print(
            f"{group:<25} "
            f"n = {len(values):,} | "
            f"Mean = ${mean_value:,.2f} | "
            f"Median = ${median_value:,.2f}"
        )

    valid_groups = {
        name: values for name, values in groups.items()
        if len(values) >= 2
    }

    welch_stat = np.nan
    welch_p = np.nan
    kruskal_stat = np.nan
    kruskal_p = np.nan

    if len(valid_groups) >= 2:
        welch_result = anova_oneway(
            list(valid_groups.values()),
            use_var="unequal"
        )
        welch_stat = welch_result.statistic
        welch_p = welch_result.pvalue

        kruskal_result = stats.kruskal(*valid_groups.values())
        kruskal_stat = kruskal_result.statistic
        kruskal_p = kruskal_result.pvalue

    overall_tests = pd.DataFrame({
        "Test": ["Welch's ANOVA", "Kruskal-Wallis"],
        "Statistic": [welch_stat, kruskal_stat],
        "p-value": [welch_p, kruskal_p]
    })

    print("\nOVERALL STATISTICAL TESTS")
    display(overall_tests)

    pairwise_results = []

    for group1, group2 in combinations(group_order, 2):
        x = groups[group1]
        y = groups[group2]

        if len(x) > 0 and len(y) > 0:
            statistic, p_value = stats.mannwhitneyu(
                x, y, alternative="two-sided"
            )
        else:
            statistic, p_value = np.nan, np.nan

        pairwise_results.append({
            "Group 1": group1,
            "Group 2": group2,
            "U Statistic": statistic,
            "Raw p-value": p_value
        })

    pairwise = pd.DataFrame(pairwise_results)
    pairwise["Adjusted p-value"] = np.nan

    valid_mask = pairwise["Raw p-value"].notna()
    if valid_mask.any():
        pairwise.loc[valid_mask, "Adjusted p-value"] = multipletests(
            pairwise.loc[valid_mask, "Raw p-value"],
            method="holm"
        )[1]

    pairwise["Result"] = np.where(
        pairwise["Adjusted p-value"] < 0.05,
        "Significant",
        "Not Significant"
    )
    pairwise.loc[
        pairwise["Adjusted p-value"].isna(), "Result"
    ] = "Not available"

    print("\nPAIRWISE MANN-WHITNEY U TESTS")
    display(pairwise)

    excel_file = os.path.join(
        OUTPUT_DIR, "01_autograph_patch_price_analysis.xlsx"
    )

    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        overall_tests.to_excel(
            writer, sheet_name="Overall Tests", index=False
        )
        pairwise.to_excel(
            writer, sheet_name="Pairwise Tests", index=False
        )

    print(f"Saved graph: {graph_file}")
    print(f"Saved Excel: {excel_file}")

    return {
        "graph": graph_file,
        "excel": excel_file,
        "summary": summary,
        "overall_tests": overall_tests,
        "pairwise": pairwise
    }

try:
    analysis_01 = run_analysis_01(df_raw)
    generated_files.extend([
        analysis_01["graph"],
        analysis_01["excel"]
    ])
    analysis_status.append([1, "Autograph & Patch", "Completed", ""])
except Exception as e:
    analysis_01 = None
    analysis_status.append([1, "Autograph & Patch", "Skipped/Error", str(e)])
    print(f"Analysis 1 skipped: {e}")



# Analysis 2 — Panini vs Topps by Card Type

Compares Panini and Topps within the same four card-feature groups.

- Panini graph color: `#FFF442`
- Topps graph color: `#D71921`
- Welch's two-sample t-test is calculated within each card type.
- Holm correction adjusts for multiple testing.
- Mann-Whitney U is included as a robustness check.


In [ ]:

def run_analysis_02(df_raw):
    analysis_name = "Analysis 2 - Panini vs Topps by Card Type"
    df = df_raw.copy()

    require_columns(
        df,
        ["brand", "asking_price_usd", "autograph", "patch"],
        analysis_name
    )

    df["asking_price_usd"] = pd.to_numeric(
        df["asking_price_usd"], errors="coerce"
    )

    df["brand_clean"] = (
        df["brand"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["autograph"] = to_bool(df["autograph"])
    df["patch"] = to_bool(df["patch"])

    df = df[
        df["brand_clean"].isin(["panini", "topps"]) &
        df["asking_price_usd"].notna() &
        (df["asking_price_usd"] > 0)
    ].copy()

    df["Brand"] = df["brand_clean"].map({
        "panini": "Panini",
        "topps": "Topps"
    })

    conditions = [
        (df["autograph"] == True) & (df["patch"] == True),
        (df["autograph"] == True) & (df["patch"] == False),
        (df["autograph"] == False) & (df["patch"] == True)
    ]

    choices = [
        "Autograph + Patch",
        "Autograph Only",
        "Patch Only"
    ]

    group_order = [
        "No Autograph / Patch",
        "Autograph Only",
        "Patch Only",
        "Autograph + Patch"
    ]

    df["card_type"] = np.select(
        conditions,
        choices,
        default="No Autograph / Patch"
    )

    df["card_type"] = pd.Categorical(
        df["card_type"],
        categories=group_order,
        ordered=True
    )

    summary = (
        df.groupby(
            ["card_type", "Brand"],
            observed=True
        )["asking_price_usd"]
        .agg(
            Listings="count",
            Average_Price="mean",
            Median_Price="median",
            Std_Dev="std"
        )
        .reset_index()
    )

    summary[
        ["Average_Price", "Median_Price", "Std_Dev"]
    ] = summary[
        ["Average_Price", "Median_Price", "Std_Dev"]
    ].round(2)

    print("SUMMARY STATISTICS")
    display(summary)

    plot_data = (
        df.groupby(
            ["card_type", "Brand"],
            observed=True
        )["asking_price_usd"]
        .mean()
        .unstack()
        .reindex(group_order)
    )

    for col in ["Panini", "Topps"]:
        if col not in plot_data.columns:
            plot_data[col] = np.nan

    x = np.arange(len(group_order))
    width = 0.36

    fig, ax = plt.subplots(figsize=(12, 6.5))

    bars_panini = ax.bar(
        x - width / 2,
        plot_data["Panini"],
        width,
        label="Panini",
        color="#FFF442",
        edgecolor="black"
    )

    bars_topps = ax.bar(
        x + width / 2,
        plot_data["Topps"],
        width,
        label="Topps",
        color="#D71921"
    )

    for bars in [bars_panini, bars_topps]:
        for bar in bars:
            height = bar.get_height()
            if pd.notna(height):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    height,
                    f"${height:,.0f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

    ax.set_title(
        "Average Asking Price by Card Type: Panini vs Topps",
        fontsize=17,
        fontweight="bold",
        pad=18
    )
    ax.set_ylabel("Average Asking Price (USD)")
    ax.set_xlabel("")
    ax.set_xticks(x)
    ax.set_xticklabels(group_order)
    ax.legend(title="Brand")
    ax.grid(axis="y", alpha=0.2)

    max_value = np.nanmax(plot_data[["Panini", "Topps"]].to_numpy())
    if np.isfinite(max_value):
        ax.set_ylim(0, max_value * 1.30)

    plt.tight_layout()

    graph_file = os.path.join(
        OUTPUT_DIR, "02_panini_vs_topps_by_card_type.png"
    )
    plt.savefig(graph_file, dpi=500, bbox_inches="tight")
    plt.show()

    test_rows = []

    for card_type in group_order:
        panini = df.loc[
            (df["card_type"] == card_type) &
            (df["Brand"] == "Panini"),
            "asking_price_usd"
        ].dropna()

        topps = df.loc[
            (df["card_type"] == card_type) &
            (df["Brand"] == "Topps"),
            "asking_price_usd"
        ].dropna()

        if len(panini) >= 2 and len(topps) >= 2:
            t_stat, p_value = stats.ttest_ind(
                panini, topps, equal_var=False
            )
        else:
            t_stat, p_value = np.nan, np.nan

        if len(panini) > 0 and len(topps) > 0:
            u_stat, mw_pvalue = stats.mannwhitneyu(
                panini, topps, alternative="two-sided"
            )
        else:
            u_stat, mw_pvalue = np.nan, np.nan

        test_rows.append({
            "Card Type": card_type,
            "Panini n": len(panini),
            "Topps n": len(topps),
            "Panini Average": panini.mean(),
            "Topps Average": topps.mean(),
            "t-statistic": t_stat,
            "p-value": p_value,
            "U Statistic": u_stat,
            "Mann-Whitney p-value": mw_pvalue
        })

    test_results = pd.DataFrame(test_rows)
    test_results["Adjusted p-value"] = np.nan

    valid_mask = test_results["p-value"].notna()
    if valid_mask.any():
        test_results.loc[valid_mask, "Adjusted p-value"] = multipletests(
            test_results.loc[valid_mask, "p-value"],
            method="holm"
        )[1]

    test_results["Result"] = np.where(
        test_results["Adjusted p-value"] < 0.05,
        "Significant",
        "Not Significant"
    )
    test_results.loc[
        test_results["Adjusted p-value"].isna(),
        "Result"
    ] = "Not available"

    print("\nSTATISTICAL TEST RESULTS")
    display(test_results)

    print("\nKEY FINDINGS")
    print("=" * 70)

    for _, row in test_results.iterrows():
        print(
            f"{row['Card Type']}: "
            f"Panini ${row['Panini Average']:,.2f} vs "
            f"Topps ${row['Topps Average']:,.2f} | "
            f"Welch p = {format_pvalue(row['p-value'])} | "
            f"Holm-adjusted p = {format_pvalue(row['Adjusted p-value'])}"
        )

    excel_file = os.path.join(
        OUTPUT_DIR, "02_panini_vs_topps_card_type_analysis.xlsx"
    )

    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        test_results.to_excel(
            writer, sheet_name="Statistical Tests", index=False
        )

    print(f"Saved graph: {graph_file}")
    print(f"Saved Excel: {excel_file}")

    return {
        "graph": graph_file,
        "excel": excel_file,
        "summary": summary,
        "tests": test_results
    }

try:
    analysis_02 = run_analysis_02(df_raw)
    generated_files.extend([
        analysis_02["graph"],
        analysis_02["excel"]
    ])
    analysis_status.append([2, "Panini vs Topps by Card Type", "Completed", ""])
except Exception as e:
    analysis_02 = None
    analysis_status.append([
        2, "Panini vs Topps by Card Type", "Skipped/Error", str(e)
    ])
    print(f"Analysis 2 skipped: {e}")



# Analysis 3 — Graded vs Ungraded

Compares average asking prices after matching the **same player and same box set**.

The main statistical test is a paired t-test across matched `Player_Canonical × product_line` groups.


In [ ]:

def run_analysis_03(df_raw):
    analysis_name = "Analysis 3 - Graded vs Ungraded"
    df = df_raw.copy()

    required_columns = [
        "Player_Canonical",
        "product_line",
        "asking_price_usd",
        "graded"
    ]
    require_columns(df, required_columns, analysis_name)

    df["asking_price_usd"] = pd.to_numeric(
        df["asking_price_usd"], errors="coerce"
    )

    graded_clean = (
        df["graded"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["graded_status"] = np.where(
        graded_clean.isin(["true", "1", "yes", "graded"]),
        "Graded",
        "Ungraded"
    )

    analysis_df = df.dropna(
        subset=[
            "Player_Canonical",
            "product_line",
            "asking_price_usd"
        ]
    ).copy()

    analysis_df = analysis_df[
        analysis_df["asking_price_usd"] > 0
    ].copy()

    grouped = (
        analysis_df
        .groupby(
            ["Player_Canonical", "product_line", "graded_status"],
            as_index=False
        )
        .agg(
            average_price=("asking_price_usd", "mean"),
            median_price=("asking_price_usd", "median"),
            listing_count=("asking_price_usd", "size")
        )
    )

    paired = (
        grouped
        .pivot(
            index=["Player_Canonical", "product_line"],
            columns="graded_status",
            values="average_price"
        )
        .reset_index()
    )

    for col in ["Graded", "Ungraded"]:
        if col not in paired.columns:
            paired[col] = np.nan

    paired = paired.dropna(
        subset=["Graded", "Ungraded"]
    ).copy()

    if len(paired) < 2:
        raise ValueError(
            "Not enough matched Player × Box Set groups "
            "for a paired t-test."
        )

    graded_mean = paired["Graded"].mean()
    ungraded_mean = paired["Ungraded"].mean()
    graded_median = paired["Graded"].median()
    ungraded_median = paired["Ungraded"].median()

    difference = graded_mean - ungraded_mean
    pct_difference = (
        difference / ungraded_mean * 100
        if ungraded_mean != 0 else np.nan
    )

    t_stat, p_value = ttest_rel(
        paired["Graded"],
        paired["Ungraded"],
        nan_policy="omit"
    )

    summary = pd.DataFrame({
        "Card Type": ["Graded", "Ungraded"],
        "Average Price (USD)": [graded_mean, ungraded_mean],
        "Median Price (USD)": [graded_median, ungraded_median],
        "Matched Groups": [len(paired), len(paired)]
    })

    stats_output = pd.DataFrame({
        "Metric": [
            "Difference (Graded - Ungraded)",
            "Percentage Difference",
            "t-statistic",
            "p-value",
            "Matched Groups"
        ],
        "Value": [
            difference,
            pct_difference,
            t_stat,
            p_value,
            len(paired)
        ]
    })

    print("SUMMARY STATISTICS")
    display(summary.style.format({
        "Average Price (USD)": "${:,.2f}",
        "Median Price (USD)": "${:,.2f}"
    }))

    print("\nPAIRED T-TEST")
    print("=" * 60)
    print(f"Average Graded Price   : ${graded_mean:,.2f}")
    print(f"Average Ungraded Price : ${ungraded_mean:,.2f}")
    print(f"Difference             : ${difference:,.2f}")
    print(f"Percentage Difference  : {pct_difference:,.2f}%")
    print(f"t-statistic            : {t_stat:.4f}")
    print(f"p-value                : {p_value:.8f}")
    print(f"Matched Groups         : {len(paired):,}")

    plot_df = pd.DataFrame({
        "Card Type": ["Ungraded", "Graded"],
        "Average Asking Price": [ungraded_mean, graded_mean]
    })

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(
        plot_df["Card Type"],
        plot_df["Average Asking Price"],
        width=0.6
    )

    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"${height:,.2f}",
            ha="center",
            va="bottom",
            fontsize=13,
            fontweight="bold"
        )

    ax.set_title(
        "Average Asking Price: Graded vs Ungraded\n"
        "Same Player and Box Set",
        fontsize=15,
        fontweight="bold",
        pad=16
    )
    ax.set_ylabel("Average Asking Price (USD)", fontsize=11)
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.text(
        0.5,
        -0.14,
        f"Matched Player × Box Set groups: {len(paired):,}",
        transform=ax.transAxes,
        ha="center",
        fontsize=10
    )

    plt.tight_layout()

    graph_file = os.path.join(
        OUTPUT_DIR,
        "03_graded_vs_ungraded_same_player_boxset.png"
    )
    plt.savefig(graph_file, dpi=500, bbox_inches="tight")
    plt.show()

    excel_file = os.path.join(
        OUTPUT_DIR,
        "03_graded_vs_ungraded_analysis.xlsx"
    )

    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        stats_output.to_excel(
            writer, sheet_name="Statistical Test", index=False
        )
        paired.to_excel(
            writer, sheet_name="Matched Groups", index=False
        )
        grouped.to_excel(
            writer, sheet_name="Grouped Data", index=False
        )

    print(f"Saved graph: {graph_file}")
    print(f"Saved Excel: {excel_file}")

    return {
        "graph": graph_file,
        "excel": excel_file,
        "summary": summary,
        "stats": stats_output,
        "paired": paired
    }

try:
    analysis_03 = run_analysis_03(df_raw)
    generated_files.extend([
        analysis_03["graph"],
        analysis_03["excel"]
    ])
    analysis_status.append([3, "Graded vs Ungraded", "Completed", ""])
except Exception as e:
    analysis_03 = None
    analysis_status.append([
        3, "Graded vs Ungraded", "Skipped/Error", str(e)
    ])
    print(f"Analysis 3 skipped: {e}")



# Analysis 4 — Rookie vs Non-Rookie

Compares Rookie and Non-Rookie cards after matching the **same player and same box set**.

Outputs include a paired t-test and Wilcoxon signed-rank test.


In [ ]:

def run_analysis_04(df_raw):
    analysis_name = "Analysis 4 - Rookie vs Non-Rookie"
    df = df_raw.copy()

    required_cols = [
        "Player_Canonical",
        "product_line",
        "asking_price_usd",
        "rookie"
    ]
    require_columns(df, required_cols, analysis_name)

    data = df[required_cols].copy()

    data["asking_price_usd"] = pd.to_numeric(
        data["asking_price_usd"],
        errors="coerce"
    )

    data = data.dropna(
        subset=[
            "Player_Canonical",
            "product_line",
            "asking_price_usd",
            "rookie"
        ]
    )

    data["rookie"] = (
        data["rookie"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False
        })
    )

    data = data.dropna(subset=["rookie"])

    group_avg = (
        data
        .groupby(
            ["Player_Canonical", "product_line", "rookie"]
        )["asking_price_usd"]
        .mean()
        .unstack("rookie")
        .rename(columns={
            False: "Non-Rookie",
            True: "Rookie"
        })
        .reset_index()
    )

    for col in ["Non-Rookie", "Rookie"]:
        if col not in group_avg.columns:
            group_avg[col] = np.nan

    group_avg = group_avg.dropna(
        subset=["Non-Rookie", "Rookie"]
    ).copy()

    if len(group_avg) < 2:
        raise ValueError(
            "Not enough matched Player + Box Set groups "
            "for paired statistical testing."
        )

    rookie_avg = group_avg["Rookie"].mean()
    nonrookie_avg = group_avg["Non-Rookie"].mean()
    rookie_median = group_avg["Rookie"].median()
    nonrookie_median = group_avg["Non-Rookie"].median()

    difference = rookie_avg - nonrookie_avg
    premium_pct = (
        (rookie_avg / nonrookie_avg - 1) * 100
        if nonrookie_avg != 0 else np.nan
    )

    summary = pd.DataFrame({
        "Metric": [
            "Matched Player + Box Set groups",
            "Average Rookie price",
            "Average Non-Rookie price",
            "Median Rookie price",
            "Median Non-Rookie price",
            "Average price difference",
            "Rookie price difference (%)"
        ],
        "Value": [
            len(group_avg),
            rookie_avg,
            nonrookie_avg,
            rookie_median,
            nonrookie_median,
            difference,
            premium_pct
        ]
    })

    t_stat, p_value = ttest_rel(
        group_avg["Rookie"],
        group_avg["Non-Rookie"]
    )

    try:
        w_stat, w_pvalue = wilcoxon(
            group_avg["Rookie"],
            group_avg["Non-Rookie"]
        )
    except ValueError:
        w_stat, w_pvalue = np.nan, np.nan

    stats_output = pd.DataFrame({
        "Test": [
            "Paired t-test",
            "Wilcoxon signed-rank test"
        ],
        "Statistic": [t_stat, w_stat],
        "p_value": [p_value, w_pvalue]
    })

    print("SUMMARY STATISTICS")
    display(summary)

    print("\nSTATISTICAL TESTS")
    display(stats_output)

    plot_df = pd.DataFrame({
        "Card Type": ["Non-Rookie", "Rookie"],
        "Average Asking Price (USD)": [
            nonrookie_avg, rookie_avg
        ]
    })

    fig, ax = plt.subplots(figsize=(8, 5.5))

    bars = ax.bar(
        plot_df["Card Type"],
        plot_df["Average Asking Price (USD)"]
    )

    ax.set_title(
        "Average Asking Price: Rookie vs Non-Rookie\n"
        "Matched by Same Player and Same Box Set",
        fontsize=14,
        pad=14
    )

    ax.set_ylabel("Average Asking Price (USD)")
    ax.set_xlabel("")

    for bar, value in zip(
        bars,
        plot_df["Average Asking Price (USD)"]
    ):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${value:,.2f}",
            ha="center",
            va="bottom",
            fontsize=11
        )

    plt.tight_layout()

    graph_file = os.path.join(
        OUTPUT_DIR,
        "04_rookie_vs_nonrookie_same_player_boxset.png"
    )
    plt.savefig(graph_file, dpi=500, bbox_inches="tight")
    plt.show()

    excel_file = os.path.join(
        OUTPUT_DIR,
        "04_rookie_vs_nonrookie_analysis.xlsx"
    )

    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        stats_output.to_excel(
            writer, sheet_name="Statistical Tests", index=False
        )
        group_avg.to_excel(
            writer, sheet_name="Matched Groups", index=False
        )

    print(f"Saved graph: {graph_file}")
    print(f"Saved Excel: {excel_file}")

    return {
        "graph": graph_file,
        "excel": excel_file,
        "summary": summary,
        "stats": stats_output,
        "matched": group_avg
    }

try:
    analysis_04 = run_analysis_04(df_raw)
    generated_files.extend([
        analysis_04["graph"],
        analysis_04["excel"]
    ])
    analysis_status.append([4, "Rookie vs Non-Rookie", "Completed", ""])
except Exception as e:
    analysis_04 = None
    analysis_status.append([
        4, "Rookie vs Non-Rookie", "Skipped/Error", str(e)
    ])
    print(f"Analysis 4 skipped: {e}")



# Analysis 5 — Asking Price by Product Tier

Compares similar Panini and Topps product tiers:

| Tier | Panini | Topps |
|---|---|---|
| Core | Prizm | Chrome |
| Mid | Select | Merlin |
| Luxury | Immaculate | Dynasty |

Both Welch's independent t-test and Mann-Whitney U are reported.


In [ ]:

def run_analysis_05(df_raw):
    analysis_name = "Analysis 5 - Price by Product Tier"
    df = df_raw.copy()

    require_columns(
        df,
        ["brand", "product_line", "asking_price_usd"],
        analysis_name
    )

    df["brand"] = (
        df["brand"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    df["product_line"] = (
        df["product_line"]
        .astype(str)
        .str.strip()
    )

    df["asking_price_usd"] = pd.to_numeric(
        df["asking_price_usd"],
        errors="coerce"
    )

    tier_mapping = {
        "Core": {
            "Panini": "Prizm",
            "Topps": "Chrome"
        },
        "Mid": {
            "Panini": "Select",
            "Topps": "Merlin"
        },
        "Luxury": {
            "Panini": "Immaculate",
            "Topps": "Dynasty"
        }
    }

    results = []

    for tier, products in tier_mapping.items():
        panini_set = products["Panini"]
        topps_set = products["Topps"]

        panini_price = df.loc[
            (df["brand"] == "panini") &
            (df["product_line"] == panini_set),
            "asking_price_usd"
        ].dropna()

        topps_price = df.loc[
            (df["brand"] == "topps") &
            (df["product_line"] == topps_set),
            "asking_price_usd"
        ].dropna()

        if len(panini_price) >= 2 and len(topps_price) >= 2:
            t_stat, t_pvalue = ttest_ind(
                panini_price,
                topps_price,
                equal_var=False
            )
        else:
            t_stat, t_pvalue = np.nan, np.nan

        if len(panini_price) > 0 and len(topps_price) > 0:
            u_stat, u_pvalue = mannwhitneyu(
                panini_price,
                topps_price,
                alternative="two-sided"
            )
        else:
            u_stat, u_pvalue = np.nan, np.nan

        results.append({
            "Tier": tier,
            "Panini Set": panini_set,
            "Panini Listings": len(panini_price),
            "Panini Mean": panini_price.mean(),
            "Panini Median": panini_price.median(),
            "Topps Set": topps_set,
            "Topps Listings": len(topps_price),
            "Topps Mean": topps_price.mean(),
            "Topps Median": topps_price.median(),
            "Mean Difference": (
                panini_price.mean() - topps_price.mean()
            ),
            "t-statistic": t_stat,
            "t-test p-value": t_pvalue,
            "U-statistic": u_stat,
            "Mann-Whitney p-value": u_pvalue
        })

    results_df = pd.DataFrame(results)

    print("SUMMARY + STATISTICAL TESTS")
    display(
        results_df.round({
            "Panini Mean": 2,
            "Panini Median": 2,
            "Topps Mean": 2,
            "Topps Median": 2,
            "Mean Difference": 2,
            "t-statistic": 4,
            "t-test p-value": 6,
            "U-statistic": 2,
            "Mann-Whitney p-value": 6
        })
    )

    graph_rows = []

    for tier, products in tier_mapping.items():
        for brand, product in products.items():
            prices = df.loc[
                (df["brand"] == brand.lower()) &
                (df["product_line"] == product),
                "asking_price_usd"
            ].dropna()

            graph_rows.append({
                "Tier": tier,
                "Brand": brand,
                "Box Set": product,
                "Listings": len(prices),
                "Average Price": prices.mean(),
                "Median Price": prices.median()
            })

    graph_df = pd.DataFrame(graph_rows)
    print("\nGRAPH DATA")
    display(graph_df.round(2))

    tier_order = ["Core", "Mid", "Luxury"]
    x = np.arange(len(tier_order))
    width = 0.34

    panini_values = []
    topps_values = []
    panini_names = []
    topps_names = []

    for tier in tier_order:
        panini_row = graph_df[
            (graph_df["Tier"] == tier) &
            (graph_df["Brand"] == "Panini")
        ].iloc[0]

        topps_row = graph_df[
            (graph_df["Tier"] == tier) &
            (graph_df["Brand"] == "Topps")
        ].iloc[0]

        panini_values.append(panini_row["Average Price"])
        topps_values.append(topps_row["Average Price"])
        panini_names.append(panini_row["Box Set"])
        topps_names.append(topps_row["Box Set"])

    fig, ax = plt.subplots(figsize=(11, 6))

    bars_panini = ax.bar(
        x - width / 2,
        panini_values,
        width,
        label="Panini",
        color="#FFF442",
        edgecolor="black"
    )

    bars_topps = ax.bar(
        x + width / 2,
        topps_values,
        width,
        label="Topps",
        color="#D71921",
        edgecolor="black"
    )

    for bars in [bars_panini, bars_topps]:
        for bar in bars:
            height = bar.get_height()
            if pd.notna(height):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    height,
                    f"${height:,.0f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

    for i in range(len(tier_order)):
        ax.text(
            x[i] - width / 2,
            -0.055,
            panini_names[i],
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=10
        )

        ax.text(
            x[i] + width / 2,
            -0.055,
            topps_names[i],
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=10
        )

    ax.set_title(
        "Average Asking Price by Product Tier",
        fontsize=16,
        fontweight="bold",
        pad=15
    )
    ax.set_ylabel("Average Asking Price (USD)")
    ax.set_xlabel("Product Tier", labelpad=28)
    ax.set_xticks(x)
    ax.set_xticklabels(
        tier_order,
        fontsize=11,
        fontweight="bold"
    )
    ax.legend(title="Brand", frameon=False)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    graph_file = os.path.join(
        OUTPUT_DIR,
        "05_average_asking_price_by_tier.png"
    )
    plt.savefig(graph_file, dpi=500, bbox_inches="tight")
    plt.show()

    print("\nWELCH'S T-TEST RESULTS")
    print("=" * 70)

    for _, row in results_df.iterrows():
        print(
            f"{row['Tier']}: "
            f"{row['Panini Set']} vs {row['Topps Set']}"
        )
        print(
            f"Panini average: ${row['Panini Mean']:,.2f}"
        )
        print(
            f"Topps average: ${row['Topps Mean']:,.2f}"
        )
        print(
            f"p-value: {format_pvalue(row['t-test p-value'])}"
        )
        print("-" * 70)

    excel_file = os.path.join(
        OUTPUT_DIR,
        "05_tier_price_statistical_results.xlsx"
    )

    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        results_df.to_excel(
            writer, sheet_name="Statistical Results", index=False
        )
        graph_df.to_excel(
            writer, sheet_name="Graph Data", index=False
        )

    print(f"Saved graph: {graph_file}")
    print(f"Saved Excel: {excel_file}")

    return {
        "graph": graph_file,
        "excel": excel_file,
        "results": results_df,
        "graph_data": graph_df
    }

try:
    analysis_05 = run_analysis_05(df_raw)
    generated_files.extend([
        analysis_05["graph"],
        analysis_05["excel"]
    ])
    analysis_status.append([5, "Price by Product Tier", "Completed", ""])
except Exception as e:
    analysis_05 = None
    analysis_status.append([
        5, "Price by Product Tier", "Skipped/Error", str(e)
    ])
    print(f"Analysis 5 skipped: {e}")



# Final Output Summary and ZIP Download

This section shows which analyses completed, lists all generated files, creates one ZIP archive, and downloads it.


In [ ]:

status_df = pd.DataFrame(
    analysis_status,
    columns=["No.", "Analysis", "Status", "Note"]
)

display(status_df)

print("\nGENERATED FILES")
print("=" * 70)

for file_path in generated_files:
    if os.path.exists(file_path):
        print(f"- {file_path}")

zip_base = "football_card_analysis_all_outputs"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=OUTPUT_DIR
)

print(f"\nZIP created: {zip_path}")
print("The ZIP contains all successfully generated PNG and Excel outputs.")

files.download(zip_path)
